In [1]:
import pandas as pd

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/karlaazuniga/etl-data-pipeline2516662022/refs/heads/main/data/raw/A_estudiantes.csv"

df = pd.read_csv(url)

df.head()

,id_estudiante,nombre,edad,correo
0,E1000,Raúl Gómez,26,carlos.castro45@correo.sv
1,E1001,Ana Castro,18,adriana.santos43@gmail.com
2,E1002,Ricardo Vásquez,23,maria.castro23@empresa.com
3,E1003,Sofía Gómez,27,luis.gomez77@correo.sv
4,E1004,Adriana Santos,26,elena.morales56@gmail.com


In [3]:
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 188 entries, 0 to 187
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_estudiante  178 non-null    object
 1   nombre         188 non-null    object
 2   edad           188 non-null    object
 3   correo         188 non-null    object
dtypes: object(4)
memory usage: 6.0+ KB


,0
id_estudiante,10
nombre,0
edad,0
correo,0


In [4]:
estudiantes = df.copy()

# limpiar nombres de columnas
estudiantes.columns = estudiantes.columns.str.strip().str.lower()

# limpiar espacios en texto
for col in estudiantes.select_dtypes(include='object').columns:
    estudiantes[col] = estudiantes[col].astype(str).str.strip()

# convertir vacíos en null
estudiantes = estudiantes.replace(r'^\s*$', pd.NA, regex=True)

# eliminar duplicados
estudiantes = estudiantes.drop_duplicates()

In [5]:
estudiantes['edad'] = pd.to_numeric(
    estudiantes['edad'],
    errors='coerce'
)

In [6]:
estudiantes['correo'] = estudiantes['correo'].astype(str)

In [7]:
validos = estudiantes[
    estudiantes['correo'].notna() &
    estudiantes['edad'].notna()
].copy()

rechazados = estudiantes[
    estudiantes['correo'].isna() |
    estudiantes['edad'].isna()
].copy()

In [8]:
def motivo(row):
    motivos = []

    if pd.isna(row['correo']):
        motivos.append("correo_invalido")

    if pd.isna(row['edad']):
        motivos.append("edad_invalido")

    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

In [9]:
validos.to_csv("estudiantes_curated.csv", index=False)
rechazados.to_csv("estudiantes_rejects.csv", index=False)